# Loan Approval Prediction

## Project Information
- **Data Source**: Loan Approval Prediction Dataset (Kaggle)
- **Objective**: Predict loan approval status using applicant features
- **Models Used**: Logistic Regression, XGBoost
- **Best Accuracy**: 83.17% (XGBoost)

## Dataset
The dataset contains information about loan applicants including:
- Personal information (Gender, Marital Status, Dependents)
- Education and employment status
- Income details (Applicant and Co-applicant)
- Loan details (Amount, Term, Credit History)
- Property area type
- Target: Loan_Status (Y/N)


In [ ]:
%pip install -r ../requirements.txt


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)


In [ ]:
# Load and explore the dataset
df = pd.read_csv('LoanApprovalPrediction.csv')
print(f"Dataset shape: {df.shape}")
df.info()
df.head()


In [ ]:
# Data preprocessing
# Drop Loan_ID as it's not a feature
df = df.drop(columns=['Loan_ID'])

# Handle missing values
print(f"Missing values before dropna: {df.isnull().sum().sum()}")
df.dropna(inplace=True)
print(f"Dataset shape after dropna: {df.shape}")

# Identify categorical columns
cat_cols = []
for col in df.select_dtypes(exclude='number').columns:
    if col != 'Loan_Status':
        print(f"{col}: {df[col].unique()}")
        cat_cols.append(col)

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=cat_cols)

# Prepare features and target
X = df_encoded.drop(columns=['Loan_Status'])
df_encoded['Loan_Status'] = df_encoded['Loan_Status'].str.lower().str.strip()
y = df_encoded['Loan_Status'].map({'y': 1, 'n': 0})

print(f"\nFeatures shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")


In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


## Model 1: Logistic Regression


In [ ]:
# Train Logistic Regression model
lr_model = LogisticRegression(random_state=SEED, max_iter=1000)
lr_model.fit(X_train, y_train)

# Make predictions
lr_preds = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_preds)

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_preds))


## Model 2: XGBoost Classifier


In [ ]:
# Train XGBoost model
xgb_model = XGBClassifier(random_state=SEED, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_preds = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_preds)

print(f"XGBoost Accuracy: {xgb_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, xgb_preds))
